# Quantization Scheme Explorer

Small-multiples view of Hōgbom CLEAN image-quality metrics across the 16 quantization schemes,
faceted by **source model** (rows) × **dynamic range / visibility fraction** (columns), with the
untouched `baseline-f64` run overlaid as a dashed reference.

One function, `plot_explorer(...)`, takes the control points as plain arguments — quantization
scheme, image metric, x-axis factor, display mode (points / box / both), baseline overlay,
log-scale y-axis. Run the setup cells once, then use one cell per plot: copy a call, change the
arguments, `Shift+Enter`. No widgets, no redraw lag.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

df = pd.read_csv("metrics_summary.csv")
df = df.sort_values(["quantization_type", "sky_model", "dynamic_range", "visibilities_fraction", "seed"])
print(f"{len(df):,} runs loaded")
df.head()


## Metric & scheme metadata

Each metric carries a label, a short description, whether a log y-axis makes sense by default, and
a display precision. `QT_GROUPS` just documents how the 16 `quantization_type` values are organized
(reference / all-stage / per-stage) — copy any value from there into a `plot_explorer(...)` call
below.


In [ ]:
METRICS = {
    "cc":        dict(col="metrics.image_metrics.cross_correlation", label="Cross-correlation",
                       desc="Correlation between the restored image and the true sky model.",
                       log=False, decimals=3),
    "snr":       dict(col="metrics.image_metrics.snr", label="SNR",
                       desc="Signal-to-noise ratio of the restored image.",
                       log=False, decimals=2),
    "psnr":      dict(col="metrics.image_metrics.psnr", label="PSNR (dB)",
                       desc="Peak signal-to-noise ratio of the restored image, in dB.",
                       log=False, decimals=2),
    "flux":      dict(col="metrics.image_metrics.flux_ratio", label="Flux ratio",
                       desc="Recovered flux / true flux (1.0 = exact recovery).",
                       log=False, decimals=3),
    "rms":       dict(col="metrics.image_metrics.residual_rms", label="Residual RMS",
                       desc="RMS of the residual image after CLEAN.",
                       log=True, decimals=4),
    "rms_sigma": dict(col="metrics.image_metrics.residual_rms_over_sigma", label="Residual RMS / σ",
                       desc="Residual RMS normalized by the expected thermal-noise sigma.",
                       log=True, decimals=1),
    "adr":       dict(col="metrics.image_metrics.achieved_dr", label="Achieved dynamic range",
                       desc="Peak flux / residual RMS actually reached by CLEAN.",
                       log=True, decimals=2),
}

QT_GROUPS = {
    "Reference":          ["baseline-f64"],
    "All stages":         ["all-bf16", "all-f16", "all-f32"],
    "Visibilities only":  ["vis-bf16", "vis-f16", "vis-f32"],
    "Dirty image only":   ["dirty-bf16", "dirty-f16", "dirty-f32"],
    "PSF only":           ["psf-bf16", "psf-f16", "psf-f32"],
    "CLEAN model only":   ["model-bf16", "model-f16", "model-f32"],
}
BASELINE_QT = "baseline-f64"
SKY_LEVELS = sorted(df["sky_model"].unique())          # rows of the facet grid
SKY_COLOR = {"gaussian": "#3a6ea5", "point": "#5b8c5a"}
# one color per quantization scheme when several are overlaid in the same plot
QT_PALETTE = ["#b8622e", "#3a6ea5", "#5b8c5a", "#8759a3", "#c2a83e", "#c14f4f", "#3f9b9b", "#c76ba3"]
BASELINE_COLOR = "#8a8f99"

print("metric_key options:", list(METRICS))
for group, items in QT_GROUPS.items():
    print(f"{group:>20}: {', '.join(items)}")


## Plotting

`_box_stats` reproduces the same convention as before: box = Q1–Q3 (linear-interpolated quantiles,
matplotlib's default), line at the median, whiskers at min/max — no separate outlier rule, which is
honest for n=5 seeds per cell. Points are placed at fixed per-seed offsets (deterministic, no jitter
randomness) so the figure is stable across re-draws.

`qt` accepts either one scheme (`"vis-f16"`) or a list of schemes (`["psf-bf16", "psf-f16",
"psf-f32"]`) — each gets its own color and its own slot within an x-position, so schemes sit
side by side instead of overplotted.


In [ ]:
def _box_stats(vals, label):
    vals = np.asarray(sorted(vals), dtype=float)
    return dict(
        med=np.percentile(vals, 50), q1=np.percentile(vals, 25), q3=np.percentile(vals, 75),
        whislo=vals.min(), whishi=vals.max(), fliers=[], label=label,
    )


def _draw_series(ax, sub, x_factor, x_levels, metric_col, color, view, dashed, width, offset,
                  show_points=True):
    """Draw one quantization scheme's boxes/points across the x_levels of one panel."""
    box_kwargs = dict(patch_artist=True, widths=width, manage_ticks=False)
    style = dict(
        boxprops=dict(facecolor=("none" if dashed else color), alpha=1 if dashed else 0.18,
                      edgecolor=color, linestyle="--" if dashed else "-", linewidth=1.3),
        medianprops=dict(color=color, linewidth=2, linestyle="--" if dashed else "-"),
        whiskerprops=dict(color=color, linewidth=1.2, linestyle="--" if dashed else "-"),
        capprops=dict(color=color, linewidth=1.2),
    )
    for i, xv in enumerate(x_levels):
        vals = sub.loc[sub[x_factor] == xv, metric_col].dropna().values
        if len(vals) == 0:
            continue
        cx = i + 1 + offset
        if view in ("both", "box"):
            ax.bxp([_box_stats(vals, "")], positions=[cx], **box_kwargs, **style)
        if view in ("both", "points") and show_points:
            seeds = sub.loc[sub[x_factor] == xv, "seed"].values
            order = np.argsort(seeds)
            n = len(vals)
            jitters = (np.arange(n) - (n - 1) / 2) * (width * 0.22)
            ax.scatter(cx + jitters[np.argsort(order)], vals, s=13, color="#222", alpha=0.55,
                       zorder=5, edgecolors="none")
        if len(vals) < 5 and show_points:
            ax.annotate(f"n={len(vals)}", (cx, 1), xycoords=("data", "axes fraction"),
                        fontsize=6, color=color, ha="center", va="bottom")


def plot_explorer(qt, metric_key, x_factor="visibilities_fraction", view="both",
                   overlay_baseline=True, log_scale=None):
    qt_list = [qt] if isinstance(qt, str) else list(qt)
    add_baseline = overlay_baseline and BASELINE_QT not in qt_list
    n_series = len(qt_list) + (1 if add_baseline else 0)

    m = METRICS[metric_key]
    log_scale = m["log"] if log_scale is None else log_scale
    col_factor = "dynamic_range" if x_factor == "visibilities_fraction" else "visibilities_fraction"
    x_levels = sorted(df[x_factor].unique())
    col_levels = sorted(df[col_factor].unique())

    slot = 0.82                                    # total width budget per x position
    each_width = slot / n_series
    offsets = [(k - (n_series - 1) / 2) * each_width for k in range(n_series)]
    colors = [QT_PALETTE[k % len(QT_PALETTE)] for k in range(len(qt_list))]

    fig, axes = plt.subplots(len(SKY_LEVELS), len(col_levels), sharey=True, sharex=True,
                              figsize=(max(2.5, 1.7 + 0.6 * n_series) * len(col_levels),
                                       2.6 * len(SKY_LEVELS)),
                              squeeze=False)

    for i, sky in enumerate(SKY_LEVELS):
        for j, cv in enumerate(col_levels):
            ax = axes[i, j]
            panel = df[(df.sky_model == sky) & (df[col_factor] == cv)]
            for k, q in enumerate(qt_list):
                sub = panel[panel.quantization_type == q]
                _draw_series(ax, sub, x_factor, x_levels, m["col"], colors[k], view,
                             dashed=False, width=each_width * 0.9, offset=offsets[k])
            if add_baseline:
                base = panel[panel.quantization_type == BASELINE_QT]
                _draw_series(ax, base, x_factor, x_levels, m["col"], BASELINE_COLOR, view,
                             dashed=True, width=each_width * 0.8, offset=offsets[-1],
                             show_points=False)

            ax.set_xticks(range(1, len(x_levels) + 1))
            xt = [f"{v:.0%}" if x_factor == "visibilities_fraction" else f"{v:g}" for v in x_levels]
            ax.set_xticklabels(xt, fontsize=8)
            ax.tick_params(labelsize=8)
            if log_scale and m["log"] is not False:
                ax.set_yscale("log")
            ax.grid(axis="y", color="#e5e5e5", linewidth=0.7, zorder=0)
            if i == 0:
                label = f"DR = {cv:g}" if col_factor == "dynamic_range" else f"vf = {cv:.0%}"
                ax.set_title(label, fontsize=9.5)
            if j == 0:
                ax.set_ylabel(sky, fontsize=9.5, color=SKY_COLOR[sky], fontweight="bold")
            if i == len(SKY_LEVELS) - 1:
                ax.set_xlabel("vf" if x_factor == "visibilities_fraction" else "DR", fontsize=8)

    handles = [plt.Line2D([0], [0], color=c, lw=6, alpha=0.5, label=q) for q, c in zip(qt_list, colors)]
    if add_baseline:
        handles.append(plt.Line2D([0], [0], color=BASELINE_COLOR, lw=1.5, ls="--",
                                   label=f"{BASELINE_QT} (reference)"))
    fig.legend(handles=handles, loc="upper right", fontsize=8, frameon=False, ncol=min(3, n_series),
               bbox_to_anchor=(0.995, 1.05 if n_series > 3 else 1.02))
    fig.suptitle(f"{m['label']}  —  rows: sky model, columns: {col_factor}, x-axis: {x_factor}",
                 fontsize=11, y=1.06 if n_series > 3 else 1.04)
    fig.tight_layout()
    plt.show()


## Plots

`plot_explorer(qt, metric_key, x_factor=..., view=..., overlay_baseline=..., log_scale=...)`

| arg | values |
|---|---|
| `qt` | one `quantization_type` string, or a **list** of several, e.g. `"vis-f16"` or `["psf-bf16", "psf-f16", "psf-f32"]` — each gets its own color and sits in its own slot at each x-position |
| `metric_key` | `"cc"`, `"snr"`, `"psnr"`, `"flux"`, `"rms"`, `"rms_sigma"`, `"adr"` |
| `x_factor` | `"visibilities_fraction"` or `"dynamic_range"` (the other becomes the facet columns) |
| `view` | `"both"`, `"points"`, `"box"` |
| `overlay_baseline` | `True` / `False` — draws `baseline-f64` dashed, in the same cell |
| `log_scale` | `None` (metric default), or `True` / `False` to override |

Each cell below is one plot. Duplicate a cell (`b` then paste, or `Esc`+`c`+`v`) and change the
arguments for a new view — re-running one cell is fast since it skips the CSV load and only redraws
that figure.


In [ ]:
# compare precisions within one stage: does bf16/f16/f32 matter for the PSF?
plot_explorer(["dirty-f32", "dirty-bf16", "dirty-f16"], "flux", x_factor="dynamic_range")


In [ ]:
# compare stages at matched precision: which stage is most sensitive to f16?
plot_explorer(["vis-f16", "dirty-f16", "psf-f16", "model-f16"], "cc", x_factor="visibilities_fraction")


---
### Notes

- Boxes use linear-interpolated Q1/median/Q3 (matplotlib/numpy default) with whiskers at min/max
  — there are only 5 seeds per cell, so a 1.5×IQR outlier rule would be misleading.
- A panel annotated `n=k` (k<5) has missing seeds for that cell — those runs didn't converge
  (`converged_reason == "stall"`) and were dropped rather than imputed.
- The baseline-f64 overlay is always drawn from the *same* facet cell (same sky model, same
  dynamic range / visibility fraction), so it's a like-for-like comparison against whichever
  scheme(s) are passed as `qt`. It's skipped automatically if `baseline-f64` is already in the list.
- To save a figure instead of just displaying it, call `plt.savefig(...)` right after
  `plot_explorer(...)` in the same cell (matplotlib keeps the last figure current).
